# Tutorial: SymPy Planar and Fixed-Radius Spherical Shock Normals

This notebook mirrors the two kinematic models used in the project:

- **Planar normal** from three satellites and one bulk velocity vector.
- **Fixed-radius spherical model** from four satellites in the co-moving frame.

It is intentionally minimal: only the equations and code needed to derive and use both models.


## Outline

1. Setup
2. Planar model (symbolic + numeric)
3. Fixed-radius spherical model (symbolic + numeric)
4. Application to one random 4-satellite set


In [1]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import display, Math


In [2]:
AU_KM = 150_000_000.0
RE_KM = 6371.0
AU_OVER_RE = AU_KM / RE_KM


def unit(vec: np.ndarray) -> np.ndarray:
    vec = np.asarray(vec, dtype=float)
    n = np.linalg.norm(vec)
    if n == 0:
        raise ValueError("Zero-length vector cannot be normalized")
    return vec / n

def render_latex(expr, lhs=None):
    latex_expr = sp.latex(expr)
    if lhs is None:
        display(Math(latex_expr))
    else:
        display(Math(f"{lhs} = {latex_expr}"))


## 1) Planar Model

Equations:

$$
\Delta t_{12} = \frac{t_1 - t_2}{AU / R_e}, \qquad
\Delta t_{13} = \frac{t_1 - t_3}{AU / R_e}
$$

$$
\mathbf{n}\cdot\left(\mathbf{r}_2 - \mathbf{r}_1 - \mathbf{v}\,\Delta t_{12}\right)=0,
\qquad
\mathbf{n}\cdot\left(\mathbf{r}_3 - \mathbf{r}_1 - \mathbf{v}\,\Delta t_{13}\right)=0
$$

with $\|\mathbf{n}\|=1$.


In [3]:
# Symbolic planar solution from the same constraints

u1, u2, u3 = sp.symbols("u1 u2 u3", real=True)
w1, w2, w3 = sp.symbols("w1 w2 w3", real=True)

u = sp.Matrix([u1, u2, u3])
w = sp.Matrix([w1, w2, w3])

n_raw_sym = u.cross(w)
n_hat_sym = sp.simplify(n_raw_sym / sp.sqrt(n_raw_sym.dot(n_raw_sym)))


In [4]:
print("Planar symbolic unit normal:")
render_latex(n_hat_sym, r"\hat{\mathbf{n}}")


Planar symbolic unit normal:


<IPython.core.display.Math object>

In [5]:
def planar_normal(
    r1: np.ndarray,
    r2: np.ndarray,
    r3: np.ndarray,
    t1: datetime,
    t2: datetime,
    t3: datetime,
    v_kms: np.ndarray,
    au_over_re: float = AU_OVER_RE,
) -> dict:
    dt12 = (t1 - t2).total_seconds() / au_over_re
    dt13 = (t1 - t3).total_seconds() / au_over_re

    d12 = np.asarray(r2, dtype=float) - np.asarray(r1, dtype=float) - np.asarray(v_kms, dtype=float) * dt12
    d13 = np.asarray(r3, dtype=float) - np.asarray(r1, dtype=float) - np.asarray(v_kms, dtype=float) * dt13

    n_raw = np.cross(d12, d13)
    n_hat = unit(n_raw)

    return {
        "dt12": dt12,
        "dt13": dt13,
        "d12": d12,
        "d13": d13,
        "n_raw": n_raw,
        "normal": n_hat,
    }


## 2) Fixed-Radius Spherical Model

For 4 satellites, define the co-moving points

$$
\mathbf{p}_i = \mathbf{r}_i - \mathbf{v}\,\Delta t_i,
\qquad
\Delta t_i = \frac{t_i - t_{ref}}{AU / R_e}
$$

and enforce one sphere through all four $\mathbf{p}_i$:

$$
\|\mathbf{p}_i - \mathbf{c}\|^2 = r_0^2, \qquad i=1,2,3,4.
$$

Subtracting equations gives a linear system for center $\mathbf{c}$:

$$
2(\mathbf{p}_1-\mathbf{p}_k)\cdot\mathbf{c} = \|\mathbf{p}_1\|^2 - \|\mathbf{p}_k\|^2,
\qquad k=2,3,4.
$$

Normals at encounter times:

$$
\mathbf{n}_i = \frac{\mathbf{r}_i - (\mathbf{c} + \mathbf{v}\,\Delta t_i)}{\|\mathbf{r}_i - (\mathbf{c} + \mathbf{v}\,\Delta t_i)\|}.
$$


In [6]:
# Symbolic center/radius construction for the fixed-radius sphere model

p1x, p1y, p1z = sp.symbols("p1x p1y p1z", real=True)
p2x, p2y, p2z = sp.symbols("p2x p2y p2z", real=True)
p3x, p3y, p3z = sp.symbols("p3x p3y p3z", real=True)
p4x, p4y, p4z = sp.symbols("p4x p4y p4z", real=True)

p1 = sp.Matrix([p1x, p1y, p1z])
p2 = sp.Matrix([p2x, p2y, p2z])
p3 = sp.Matrix([p3x, p3y, p3z])
p4 = sp.Matrix([p4x, p4y, p4z])

A_sym = 2 * sp.Matrix([
    (p1 - p2).T,
    (p1 - p3).T,
    (p1 - p4).T,
])

b_sym = sp.Matrix([
    p1.dot(p1) - p2.dot(p2),
    p1.dot(p1) - p3.dot(p3),
    p1.dot(p1) - p4.dot(p4),
])

c_sym = A_sym.LUsolve(b_sym)
r0_sym = sp.sqrt((p1 - c_sym).dot(p1 - c_sym))


In [7]:
print("Sphere-through-4-points symbolic linear system:")
render_latex(A_sym, r"\mathbf{A}")
render_latex(b_sym, r"\mathbf{b}")
display(Math(r"\mathbf{c} = \mathbf{A}^{-1}\mathbf{b}"))
display(Math(r"r_0 = \sqrt{(\mathbf{p}_1-\mathbf{c})\cdot(\mathbf{p}_1-\mathbf{c})}"))


Sphere-through-4-points symbolic linear system:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
def fixed_radius_sphere_normals(
    sat_data: dict,
    v_kms: np.ndarray,
    au_over_re: float = AU_OVER_RE,
) -> dict:
    names = list(sat_data.keys())
    if len(names) != 4:
        raise ValueError("fixed_radius_sphere_normals expects exactly 4 satellites")

    t_ref = min(sat_data[name]["t"] for name in names)

    dt = {
        name: (sat_data[name]["t"] - t_ref).total_seconds() / au_over_re
        for name in names
    }

    pos = {name: np.asarray(sat_data[name]["r"], dtype=float) for name in names}
    advected = {name: pos[name] - np.asarray(v_kms, dtype=float) * dt[name] for name in names}

    p1, p2, p3, p4 = [advected[name] for name in names]

    A = 2 * np.vstack([
        p1 - p2,
        p1 - p3,
        p1 - p4,
    ])
    b = np.array([
        np.dot(p1, p1) - np.dot(p2, p2),
        np.dot(p1, p1) - np.dot(p3, p3),
        np.dot(p1, p1) - np.dot(p4, p4),
    ])

    if abs(np.linalg.det(A)) < 1e-12:
        raise ValueError("Singular geometry: cannot solve unique sphere")

    center = np.linalg.solve(A, b)
    r0 = float(np.linalg.norm(p1 - center))

    normals = {
        name: unit(pos[name] - (center + np.asarray(v_kms, dtype=float) * dt[name]))
        for name in names
    }

    return {
        "reference_time": t_ref,
        "dt": dt,
        "positions": pos,
        "advected_positions": advected,
        "center": center,
        "radius": r0,
        "normals": normals,
    }


## 3) Application to One Random 4-Satellite Set

We generate one reproducible synthetic event where the data are exactly consistent with the fixed-radius moving-sphere model, then evaluate both solvers.


In [9]:
rng = np.random.default_rng(42)

sat_names = ["ACE", "Wind", "DSCOVR", "MMS1"]
t_ref_true = datetime(2025, 6, 1, 19, 19, 0, tzinfo=timezone.utc)

time_offsets = rng.integers(-240, 241, size=4)
times = {
    name: t_ref_true + timedelta(seconds=int(offset))
    for name, offset in zip(sat_names, time_offsets)
}

v_true_kms = np.array([-420.0, 35.0, -15.0]) + rng.normal(scale=8.0, size=3)
center_true = np.array([140.0, -30.0, 45.0]) + rng.normal(scale=6.0, size=3)
r0_true = 280.0

dir_vectors = rng.normal(size=(4, 3))
dir_vectors = dir_vectors / np.linalg.norm(dir_vectors, axis=1, keepdims=True)

sat_data = {}
true_normals = {}
for name, direction in zip(sat_names, dir_vectors):
    dt_i = (times[name] - t_ref_true).total_seconds() / AU_OVER_RE
    r_i = center_true + v_true_kms * dt_i + r0_true * direction
    sat_data[name] = {"t": times[name], "r": r_i}
    true_normals[name] = direction

summary = pd.DataFrame(
    {
        "sat": sat_names,
        "t": [times[s] for s in sat_names],
        "x_re": [sat_data[s]["r"][0] for s in sat_names],
        "y_re": [sat_data[s]["r"][1] for s in sat_names],
        "z_re": [sat_data[s]["r"][2] for s in sat_names],
    }
)
summary


,sat,t,x_re,y_re,z_re
0,ACE,2025-06-01 19:15:42+00:00,131.829124,-224.527753,244.319541
1,Wind,2025-06-01 19:21:12+00:00,288.699812,-15.510326,273.126404
2,DSCOVR,2025-06-01 19:20:14+00:00,256.099757,-259.245456,141.769660
3,MMS1,2025-06-01 19:18:31+00:00,-73.610663,159.716842,32.398481


In [10]:
triad = sat_names[:3]

planar_result = planar_normal(
    r1=sat_data[triad[0]]["r"],
    r2=sat_data[triad[1]]["r"],
    r3=sat_data[triad[2]]["r"],
    t1=sat_data[triad[0]]["t"],
    t2=sat_data[triad[1]]["t"],
    t3=sat_data[triad[2]]["t"],
    v_kms=v_true_kms,
)

print("Planar triad:", triad)
print("Planar unit normal:", np.round(planar_result["normal"], 6))


Planar triad: ['ACE', 'Wind', 'DSCOVR']
Planar unit normal: [-0.500189  0.459821 -0.733741]


In [11]:
sphere_result = fixed_radius_sphere_normals(sat_data=sat_data, v_kms=v_true_kms)

print("Recovered center:", np.round(sphere_result["center"], 6))
print("True center:     ", np.round(center_true, 6))
print("Recovered r0:", round(sphere_result["radius"], 6))
print("True r0:     ", round(r0_true, 6))


Recovered center: [135.668517 -29.590577  43.359952]
True center:      [132.186923 -29.232958  43.102544]
Recovered r0: 280.0
True r0:      280.0


In [12]:
rows = []
for name in sat_names:
    n_est = sphere_result["normals"][name]
    n_true = true_normals[name]
    alignment = float(np.dot(n_est, n_true))

    p = sphere_result["advected_positions"][name]
    residual = float(np.linalg.norm(p - sphere_result["center"]) - sphere_result["radius"])

    rows.append(
        {
            "sat": name,
            "n_est_x": n_est[0],
            "n_est_y": n_est[1],
            "n_est_z": n_est[2],
            "dot(n_est, n_true)": alignment,
            "sphere_residual_re": residual,
        }
    )

pd.DataFrame(rows)


,sat,n_est_x,n_est_y,n_est_z,"dot(n_est, n_true)",sphere_residual_re
0,ACE,-0.013712,-0.696204,0.717713,1.0,0.0
1,Wind,0.567264,0.048158,0.822127,1.0,0.0
2,DSCOVR,0.447193,-0.821951,0.352726,1.0,0.0
3,MMS1,-0.736813,0.675008,-0.038363,1.0,0.0


## 4) Cross-Validate Python Models vs Wolfram Results

This section reuses the same satellite sets from `Shocks/Kinematic Shock Normals.json`,
pulls satellite states from parquet with Wolfram-like row-selection behavior, and compares
Python model outputs against Wolfram outputs with explicit numeric tolerances.


In [13]:
SHOCK_PARAMS_PATH = Path("Shocks") / "Shock Params.json"
PYTHON_KINEMATIC_NORMALS_PATH = Path("Shocks") / "Kinematic Shock Normals.json"
WOLFRAM_KINEMATIC_NORMALS_PATH = Path("Shocks") / "Kinematic Shock Normals (Wolfram).json"
DATA_ROOT = Path("Data")
DEFAULT_NAIVE_TZ = "UTC"
SAT_BLACKLIST = {"stereo", "themisc"}

shock_params = json.loads(SHOCK_PARAMS_PATH.read_text())
wolfram_normals = json.loads(WOLFRAM_KINEMATIC_NORMALS_PATH.read_text())

def iter_shock_param_events() -> list[str]:
    events = []
    for event, params in shock_params.items():
        if not isinstance(params, dict):
            warnings.warn(f"Skipping {event}: shock-params entry is not a mapping", stacklevel=2)
            continue
        events.append(event)
    return sorted(events)

def canonical_sat_name(name: str) -> str:
    return name.lower().replace("_", "").replace("-", "")

def parse_date_value(value, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> pd.Timestamp:
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        ts = ts.tz_localize(default_naive_tz)
    return ts.tz_convert("UTC")

def to_standard_rows(df: pd.DataFrame) -> list[dict]:
    std = df.reset_index().copy()

    if "time" not in std.columns:
        std["time"] = pd.NaT

    return std.to_dict("records")

def extract_row_time(row: dict, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> pd.Timestamp:
    return parse_date_value(row["time"], default_naive_tz=default_naive_tz)

def nearest_row_like_wolfram(rows: list[dict], t0: pd.Timestamp, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> dict:
    parsed_times = [extract_row_time(row, default_naive_tz=default_naive_tz) for row in rows]
    valid_idx = [i for i, ts in enumerate(parsed_times) if not pd.isna(ts)]
    if valid_idx:
        deltas = [abs((parsed_times[i] - t0).total_seconds()) for i in valid_idx]
        best = valid_idx[int(np.argmin(deltas))]
    else:
        best = int(np.ceil(len(rows) / 2) - 1)
    return rows[best]

def _pick_position_keys(row: dict) -> tuple[str, str, str] | None:
    for keys in [
        ("X_GSE", "Y_GSE", "Z_GSE"),
        ("x_gse", "y_gse", "z_gse"),
        ("X", "Y", "Z"),
        ("x", "y", "z"),
    ]:
        if all(key in row for key in keys):
            return keys
    return None

def nearest_row_with_position_like_wolfram(rows: list[dict], t0: pd.Timestamp, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> dict | None:
    rows_with_pos = []
    for row in rows:
        pos = extract_position_from_row(row)
        if pos is not None and np.isfinite(pos).all():
            rows_with_pos.append(row)
    if not rows_with_pos:
        return None
    return nearest_row_like_wolfram(rows_with_pos, t0, default_naive_tz=default_naive_tz)

def extract_position_from_row(row: dict) -> np.ndarray | None:
    pos_keys = _pick_position_keys(row)
    if pos_keys is None:
        return None
    pos = np.asarray([row[pos_keys[0]], row[pos_keys[1]], row[pos_keys[2]]], dtype=float)
    finite = pos[np.isfinite(pos)]
    if finite.size > 0 and np.max(np.abs(finite)) > 5000:
        pos = pos / RE_KM
    return pos

def extract_velocity_from_row(row: dict) -> np.ndarray | None:
    vector_keys = [("V_X_GSE", "V_Y_GSE", "V_Z_GSE"), ("v_x", "v_y", "v_z"), ("VX", "VY", "VZ"), ("vx", "vy", "vz")]
    for keys in vector_keys:
        if all(key in row and pd.notna(row[key]) for key in keys):
            return np.asarray([row[key] for key in keys], dtype=float)

    for key in ["v", "V"]:
        if key in row and pd.notna(row[key]):
            speed = float(row[key])
            return np.asarray([-abs(speed), 0.0, 0.0], dtype=float)

    return None

def load_sat_state_for_t0(event_dir: Path, sat_name: str, t0_raw, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> dict:
    sat_map = {canonical_sat_name(pp.stem): pp for pp in event_dir.glob("*.parquet")}
    sat_file = sat_map[canonical_sat_name(sat_name)]

    rows = to_standard_rows(pd.read_parquet(sat_file))

    t0 = parse_date_value(t0_raw, default_naive_tz=default_naive_tz)
    row_t0 = nearest_row_like_wolfram(rows, t0, default_naive_tz=default_naive_tz)
    row_pos = row_t0

    pos = extract_position_from_row(row_pos)
    if pos is None:
        warnings.warn(f"Skipping {event_dir.name}/{sat_name}: no recognized position columns in {sat_file.name}", stacklevel=2)
        return {
            "status": "missing_position_columns",
            "file": str(sat_file),
            "t0": t0,
        }
    used_position_fallback = False
    if not np.isfinite(pos).all():
        fallback_row = nearest_row_with_position_like_wolfram(rows, t0, default_naive_tz=default_naive_tz)
        if fallback_row is not None:
            row_pos = fallback_row
            pos = extract_position_from_row(row_pos)
            used_position_fallback = True

    if pos is None or not np.isfinite(pos).all():
        warnings.warn(f"Skipping {event_dir.name}/{sat_name}: no usable position row in {sat_file.name}", stacklevel=2)
        return {
            "status": "missing_position",
            "file": str(sat_file),
            "t0": t0,
        }

    t0_row_time = extract_row_time(row_t0, default_naive_tz=default_naive_tz)
    pos_row_time = extract_row_time(row_pos, default_naive_tz=default_naive_tz)

    return {
        "t": t0.to_pydatetime(),
        "r": pos,
        "t0_row_time": t0_row_time,
        "position_row_time": pos_row_time,
        "t0_row_time_offset_s": float((t0_row_time - t0).total_seconds()) if not pd.isna(t0_row_time) else np.nan,
        "position_row_time_offset_s": float((pos_row_time - t0).total_seconds()) if not pd.isna(pos_row_time) else np.nan,
        "used_position_fallback": used_position_fallback,
    }

def build_event_sat_data_wolfram(event: str, sat_names: list[str] | None = None, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> dict:
    event_params = shock_params[event]
    key_map = {canonical_sat_name(k): k for k in event_params}

    if sat_names is None:
        sat_names = list(event_params.keys())

    out = {}
    for sat in sat_names:
        sat_key = key_map[canonical_sat_name(sat)]
        sat_state = load_sat_state_for_t0(
            event_dir=DATA_ROOT / event,
            sat_name=sat,
            t0_raw=event_params[sat_key]["t0"],
            default_naive_tz=default_naive_tz,
        )
        if sat_state.get("status") != "ok":
            warnings.warn(f"Skipping {event}/{sat}: status={sat_state.get('status')}", stacklevel=2)
            continue
        out[sat] = sat_state
    return out

def load_date_data(event: str, event_params: dict, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> dict:
    event_dir = DATA_ROOT / event
    sat_map = {canonical_sat_name(pp.stem): pp for pp in event_dir.glob("*.parquet")}
    out = {}

    for sat_key, sat_params in event_params.items():
        sat_norm = canonical_sat_name(sat_key)
        if sat_norm in SAT_BLACKLIST:
            out[sat_key] = {"status": "blacklisted"}
            continue

        sat_file = sat_map.get(sat_norm)
        if sat_file is None:
            out[sat_key] = {"status": "missing_file"}
            continue

        rows = to_standard_rows(pd.read_parquet(sat_file))
        t0 = parse_date_value(sat_params["t0"], default_naive_tz=default_naive_tz)

        if not rows:
            out[sat_key] = {"status": "no_rows_or_bad_t0", "file": str(sat_file), "t0": t0}
            continue

        row = nearest_row_like_wolfram(rows, t0, default_naive_tz=default_naive_tz)
        row_pos = row
        pos = extract_position_from_row(row_pos)
        if pos is None:
            warnings.warn(f"Skipping {event}/{sat_key}: no recognized position columns in {sat_file.name}", stacklevel=2)
            out[sat_key] = {"status": "missing_position_columns", "file": str(sat_file), "t0": t0}
            continue
        if not np.isfinite(pos).all():
            fallback_row = nearest_row_with_position_like_wolfram(rows, t0, default_naive_tz=default_naive_tz)
            if fallback_row is not None:
                row_pos = fallback_row
                pos = extract_position_from_row(row_pos)

        if pos is None or not np.isfinite(pos).all():
            warnings.warn(f"Skipping {event}/{sat_key}: no usable position row in {sat_file.name}", stacklevel=2)
            out[sat_key] = {"status": "missing_position", "file": str(sat_file), "t0": t0}
            continue

        velocity = extract_velocity_from_row(row)
        out[sat_key] = {
            "status": "ok",
            "file": str(sat_file),
            "t0": t0,
            "params": sat_params,
            "position": pos,
            "velocity": velocity,
            "row": row,
            "row_position": row_pos,
        }

    return out

def _valid_position_satellites(date_data: dict) -> list[str]:
    valid = []
    for sat, entry in date_data.items():
        pos = entry.get("position")
        t0 = entry.get("t0")
        if isinstance(pos, np.ndarray) and np.isfinite(pos).all() and isinstance(t0, datetime):
            valid.append(sat)
    return valid

def choose_planar_sat_triplet(date_data: dict) -> list[str] | None:
    valid = _valid_position_satellites(date_data)
    preferred_triplets = [
        ["ace", "wind", "dscovr"],
        ["ace", "wind", "mms1"],
        ["ace", "wind", "themis-b"],
        ["ace", "wind", "themis-c"],
        ["ace", "mms1", "wind"],
        ["wind", "themis-b", "themis-c"],
    ]
    valid_norm = {canonical_sat_name(s) for s in valid}

    for triplet in preferred_triplets:
        if set(map(canonical_sat_name, triplet)).issubset(valid_norm):
            return triplet

    if len(valid) >= 3:
        return valid[:3]
    return None

def choose_spherical_sat_quad(date_data: dict) -> list[str] | None:
    valid = _valid_position_satellites(date_data)
    preferred_quads = [
        ["ace", "wind", "dscovr", "mms1"],
        ["ace", "mms1", "dscovr", "themis-b"],
        ["dscovr", "ace", "wind", "themis-b"],
        ["ace", "wind", "dscovr", "mms1"],
        ["ace", "wind", "dscovr", "soho"],
    ]
    valid_norm = {canonical_sat_name(s) for s in valid}

    for quad in preferred_quads:
        if set(map(canonical_sat_name, quad)).issubset(valid_norm):
            return quad

    if len(valid) >= 4:
        return valid[:4]
    return None

def choose_velocity_vector(date_data: dict, preferred: list[str] | None = None) -> dict:
    preferred = preferred or ["dscovr", "wind", "ace"]
    candidates = preferred + [sat for sat in date_data if sat not in preferred]
    for sat in candidates:
        entry = date_data.get(sat, {})
        velocity = entry.get("velocity")
        if isinstance(velocity, np.ndarray) and np.isfinite(velocity).all():
            return {"sat": sat, "v": velocity}
    return {"sat": None, "v": None}

def planar_normal_for_date_data(date_data: dict) -> dict:
    triplet = choose_planar_sat_triplet(date_data)
    if triplet is None:
        return {"status": "insufficient_satellites"}

    vel_info = choose_velocity_vector(date_data)
    if vel_info["sat"] is None:
        return {"status": "no_velocity", "triplet": triplet}

    s1, s2, s3 = triplet
    result = planar_normal(
        r1=date_data[s1]["position"],
        r2=date_data[s2]["position"],
        r3=date_data[s3]["position"],
        t1=date_data[s1]["t0"],
        t2=date_data[s2]["t0"],
        t3=date_data[s3]["t0"],
        v_kms=vel_info["v"],
    )
    return {
        "status": "ok",
        "triplet": triplet,
        "velocity_sat": vel_info["sat"],
        "velocity": vel_info["v"],
        "normal": result["normal"],
    }

def spherical_normals_for_date_data(date_data: dict) -> dict:
    valid_sats = _valid_position_satellites(date_data)
    quad = choose_spherical_sat_quad(date_data)
    if quad is None:
        return {"status": "insufficient_satellites", "satellites": valid_sats}

    vel_info = choose_velocity_vector(date_data, preferred=["dscovr", "wind", "ace", "themis-b"])
    if vel_info["sat"] is None:
        return {"status": "no_velocity", "satellites": quad}

    sat_data = {sat: {"t": date_data[sat]["t0"], "r": date_data[sat]["position"]} for sat in quad}
    reference_sat = min(quad, key=lambda sat: date_data[sat]["t0"])
    t_ref = min(sat_data[sat]["t"] for sat in quad)
    dt = {
        sat: (sat_data[sat]["t"] - t_ref).total_seconds() / AU_OVER_RE
        for sat in quad
    }
    advected = {
        sat: np.asarray(sat_data[sat]["r"], dtype=float) - np.asarray(vel_info["v"], dtype=float) * dt[sat]
        for sat in quad
    }
    p1, p2, p3, p4 = [advected[sat] for sat in quad]
    A = 2 * np.vstack([
        p1 - p2,
        p1 - p3,
        p1 - p4,
    ])
    if abs(np.linalg.det(A)) < 1e-12:
        return {"status": "singular_geometry", "satellites": quad, "velocity_sat": vel_info["sat"], "reference_sat": reference_sat}

    fit = fixed_radius_sphere_normals(sat_data=sat_data, v_kms=vel_info["v"])
    sigma = float(fit["radius"] ** 2)

    return {
        "status": "ok",
        "method": "fixed_radius_4sat",
        "velocity_sat": vel_info["sat"],
        "reference_sat": reference_sat,
        "velocity": vel_info["v"],
        "satellites": quad,
        "center": fit["center"],
        "r0": float(fit["radius"]),
        "sigma": sigma,
        "normals": fit["normals"],
    }

def compute_date_normals(event: str, event_params: dict) -> dict:
    date_data = load_date_data(event, event_params)
    return {
        "planar": planar_normal_for_date_data(date_data),
        "spherical": spherical_normals_for_date_data(date_data),
    }

def unsigned_angle_deg(a: np.ndarray, b: np.ndarray) -> float:
    dot = float(np.clip(np.dot(unit(a), unit(b)), -1.0, 1.0))
    angle = float(np.degrees(np.arccos(dot)))
    return min(angle, 180.0 - angle)

def _json_ready(value):
    if isinstance(value, dict):
        return {key: _json_ready(val) for key, val in value.items()}
    if isinstance(value, list):
        return [_json_ready(val) for val in value]
    if isinstance(value, tuple):
        return [_json_ready(val) for val in value]
    if isinstance(value, np.ndarray):
        return [_json_ready(val) for val in value.tolist()]
    if isinstance(value, (np.floating, float)):
        return float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return pd.Timestamp(value).strftime("%Y-%m-%d %H:%M:%S")
    return value

def build_all_python_normals() -> dict:
    return {event: compute_date_normals(event, shock_params[event]) for event in iter_shock_param_events()}


In [14]:
python_normals = build_all_python_normals()
PYTHON_KINEMATIC_NORMALS_PATH.write_text(
    json.dumps(_json_ready(python_normals), indent=2, sort_keys=True)
)
print(f"Exported: {PYTHON_KINEMATIC_NORMALS_PATH}")

EVENTS_TO_CHECK = [
    event for event in iter_shock_param_events() if event in wolfram_normals
]

PLANAR_ANGLE_TOL_DEG = 1.0
PLANAR_UNSIGNED_L2_TOL = 2e-2
SPHERE_MAX_NORMAL_ANGLE_TOL_DEG = 1.0
SPHERE_CENTER_TOL_RE = 1e-1
SPHERE_RADIUS_TOL_RE = 1e-1

planar_rows = []
spherical_rows = []
spherical_sat_rows = []

for event in EVENTS_TO_CHECK:
    event_params = shock_params[event]
    event_normals = wolfram_normals[event]
    event_python_normals = python_normals[event]

    param_key_map = {canonical_sat_name(k): k for k in event_params}

    planar_wf = event_normals.get("planar", {})
    planar_py = event_python_normals.get("planar", {})
    if planar_wf.get("status") == "ok" and planar_wf.get("triplet"):
        triplet = planar_wf["triplet"]
        if all(canonical_sat_name(s) in param_key_map for s in triplet) and planar_py.get("status") == "ok":
            py_normal_vec = np.asarray(planar_py["normal"], dtype=float)
            wf_normal = np.asarray(planar_wf["normal"], dtype=float)

            unsigned_angle = unsigned_angle_deg(py_normal_vec, wf_normal)
            unsigned_l2 = float(min(np.linalg.norm(py_normal_vec - wf_normal), np.linalg.norm(py_normal_vec + wf_normal)))

            sat_data = build_event_sat_data_wolfram(event=event, sat_names=triplet)
            if not all(s in sat_data for s in triplet):
                warnings.warn(f"Skipping planar validation for {event}: incomplete satellite state after filtering", stacklevel=2)
                continue
            planar_rows.append({
                "event": event,
                "triplet": ", ".join(triplet),
                "unsigned_angle_deg": unsigned_angle,
                "unsigned_l2": unsigned_l2,
                "max_position_row_time_offset_s": float(np.nanmax([abs(sat_data[s]["position_row_time_offset_s"]) for s in triplet])),
                "used_position_fallback_count": int(sum(bool(sat_data[s]["used_position_fallback"]) for s in triplet)),
                "within_numerical_reason": bool((unsigned_angle <= PLANAR_ANGLE_TOL_DEG) and (unsigned_l2 <= PLANAR_UNSIGNED_L2_TOL)),
            })

    spherical_wf = event_normals.get("spherical", {})
    spherical_py = event_python_normals.get("spherical", {})
    sats = spherical_wf.get("satellites", [])
    if spherical_wf.get("status") == "ok" and spherical_wf.get("method") == "fixed_radius_4sat" and len(sats) == 4:
        if all(canonical_sat_name(s) in param_key_map for s in sats) and spherical_py.get("status") == "ok":
            wf_center = np.asarray(spherical_wf["center"], dtype=float)
            py_center = np.asarray(spherical_py["center"], dtype=float)
            wf_radius = float(spherical_wf["r0"])
            py_radius = float(spherical_py["r0"])
            wf_normals_map = {canonical_sat_name(k): np.asarray(v, dtype=float) for k, v in spherical_wf["normals"].items()}
            py_normals_map = {canonical_sat_name(k): np.asarray(v, dtype=float) for k, v in spherical_py["normals"].items()}

            sat_angles = []
            sat_l2 = []
            for sat in sats:
                wf_sat_normal = wf_normals_map[canonical_sat_name(sat)]
                py_sat_normal = py_normals_map[canonical_sat_name(sat)]
                angle = unsigned_angle_deg(py_sat_normal, wf_sat_normal)
                ul2 = float(min(np.linalg.norm(py_sat_normal - wf_sat_normal), np.linalg.norm(py_sat_normal + wf_sat_normal)))
                sat_angles.append(angle)
                sat_l2.append(ul2)
                spherical_sat_rows.append({
                    "event": event,
                    "sat": sat,
                    "normal_unsigned_angle_deg": angle,
                    "normal_unsigned_l2": ul2,
                })

            sat_data = build_event_sat_data_wolfram(event=event, sat_names=sats)
            if not all(s in sat_data for s in sats):
                warnings.warn(f"Skipping spherical validation for {event}: incomplete satellite state after filtering", stacklevel=2)
                continue
            center_l2 = float(np.linalg.norm(py_center - wf_center))
            radius_abs_err = float(abs(py_radius - wf_radius))

            spherical_rows.append({
                "event": event,
                "satellites": ", ".join(sats),
                "max_normal_unsigned_angle_deg": float(np.max(sat_angles)),
                "max_normal_unsigned_l2": float(np.max(sat_l2)),
                "center_l2_re": center_l2,
                "radius_abs_err_re": radius_abs_err,
                "max_position_row_time_offset_s": float(np.nanmax([abs(sat_data[s]["position_row_time_offset_s"]) for s in sats])),
                "used_position_fallback_count": int(sum(bool(sat_data[s]["used_position_fallback"]) for s in sats)),
                "within_numerical_reason": bool(
                    (np.max(sat_angles) <= SPHERE_MAX_NORMAL_ANGLE_TOL_DEG)
                    and (center_l2 <= SPHERE_CENTER_TOL_RE)
                    and (radius_abs_err <= SPHERE_RADIUS_TOL_RE)
                ),
            })

planar_validation_df = pd.DataFrame(
    planar_rows,
    columns=[
        "event",
        "triplet",
        "unsigned_angle_deg",
        "unsigned_l2",
        "max_position_row_time_offset_s",
        "used_position_fallback_count",
        "within_numerical_reason",
    ],
)
if len(planar_validation_df):
    planar_validation_df = planar_validation_df.sort_values("event").reset_index(drop=True)

spherical_validation_df = pd.DataFrame(
    spherical_rows,
    columns=[
        "event",
        "satellites",
        "max_normal_unsigned_angle_deg",
        "max_normal_unsigned_l2",
        "center_l2_re",
        "radius_abs_err_re",
        "max_position_row_time_offset_s",
        "used_position_fallback_count",
        "within_numerical_reason",
    ],
)
if len(spherical_validation_df):
    spherical_validation_df = spherical_validation_df.sort_values("event").reset_index(drop=True)

spherical_sat_validation_df = pd.DataFrame(
    spherical_sat_rows,
    columns=[
        "event",
        "sat",
        "normal_unsigned_angle_deg",
        "normal_unsigned_l2",
    ],
)
if len(spherical_sat_validation_df):
    spherical_sat_validation_df = spherical_sat_validation_df.sort_values(["event", "sat"]).reset_index(drop=True)

summary_df = pd.DataFrame([
    {
        "model": "planar",
        "checked_cases": len(planar_validation_df),
        "within_numerical_reason": int(planar_validation_df["within_numerical_reason"].sum()) if len(planar_validation_df) else 0,
    },
    {
        "model": "spherical_fixed_radius_4sat",
        "checked_cases": len(spherical_validation_df),
        "within_numerical_reason": int(spherical_validation_df["within_numerical_reason"].sum()) if len(spherical_validation_df) else 0,
    },
])

print("Planar validation")
display(planar_validation_df)
print("\nSpherical validation")
display(spherical_validation_df)
print("\nPer-satellite spherical normal comparison")
display(spherical_sat_validation_df)
print("\nSummary")
display(summary_df)


Exported: Shocks/Kinematic Shock Normals.json
Planar validation


,event,triplet,unsigned_angle_deg,unsigned_l2,max_position_row_time_offset_s,used_position_fallback_count,within_numerical_reason



Spherical validation


,event,satellites,max_normal_unsigned_angle_deg,max_normal_unsigned_l2,center_l2_re,radius_abs_err_re,max_position_row_time_offset_s,used_position_fallback_count,within_numerical_reason



Per-satellite spherical normal comparison


,event,sat,normal_unsigned_angle_deg,normal_unsigned_l2



Summary


,model,checked_cases,within_numerical_reason
0,planar,0,0
1,spherical_fixed_radius_4sat,0,0


## Notes

- The planar model uses **3 satellites** and gives one front normal.
- The fixed-radius spherical model uses **4 satellites** and returns one normal per satellite.
- For real events, data quality and near-singular geometry matter; this notebook keeps only the core model logic.
